# 19 — Bagging 5-Fold na versão One-Hot (a que realmente generaliza)
## PRT Seguros

Os experimentos `16`-`18` mostraram um padrão claro: tudo construído em cima da codificação
**categórica nativa** do CatBoost ficou pior no Kaggle público do que o CatBoost tunado original
(one-hot), mesmo com AUC de validação igual ou melhor. Ou seja, o one-hot generaliza melhor pra
essa base específica — provavelmente porque o *ordered target statistics* do CatBoost, apesar de
mais sofisticado, acaba se ajustando de um jeito que não transfere tão bem sob o shift que já
diagnosticamos.

Este notebook combina as **duas técnicas que comprovadamente ajudaram**: a base one-hot com
features `_v2` (shift-features removidas + engenharia) + bagging 5-fold (que reduz variância de
generalização).

## 1. Carregar dados `_v2` (one-hot) e juntar treino+val pra fazer 5-fold no total

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

train = pd.read_csv("dados_processados/train_model_ready_v2.csv")
val = pd.read_csv("dados_processados/val_model_ready_v2.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready_v2.csv")

treino_completo = pd.concat([train, val], ignore_index=True)
feature_cols = [c for c in treino_completo.columns if c not in (ID_COL, TARGET_COL)]

X_full = treino_completo[feature_cols]
y_full = treino_completo[TARGET_COL]
X_kaggle = kaggle[feature_cols]

print(f"Treino completo: {X_full.shape} | Kaggle: {X_kaggle.shape}")


Treino completo: (100000, 79) | Kaggle: (100000, 79)


## 2. Treinar 5 folds com os hiperparâmetros tunados do notebook `11`

In [2]:
MELHORES_PARAMS_CAT = {"random_strength": 0.5, "learning_rate": 0.02, "l2_leaf_reg": 9, "depth": 6}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_proba = np.zeros(len(X_full))
kaggle_proba_por_fold = []

for fold, (idx_tr, idx_val_fold) in enumerate(skf.split(X_full, y_full)):
    X_tr_fold, y_tr_fold = X_full.iloc[idx_tr], y_full.iloc[idx_tr]
    X_val_fold, y_val_fold = X_full.iloc[idx_val_fold], y_full.iloc[idx_val_fold]

    idx_es_tr, idx_es_val = train_test_split(
        np.arange(len(X_tr_fold)), test_size=0.15, stratify=y_tr_fold, random_state=RANDOM_STATE
    )
    neg_pos_ratio = (y_tr_fold.iloc[idx_es_tr] == 0).sum() / (y_tr_fold.iloc[idx_es_tr] == 1).sum()

    modelo_fold = CatBoostClassifier(
        iterations=3000, **MELHORES_PARAMS_CAT, scale_pos_weight=neg_pos_ratio,
        eval_metric="AUC", random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=False,
    )
    modelo_fold.fit(
        X_tr_fold.iloc[idx_es_tr], y_tr_fold.iloc[idx_es_tr],
        eval_set=(X_tr_fold.iloc[idx_es_val], y_tr_fold.iloc[idx_es_val]),
    )

    proba_fold = modelo_fold.predict_proba(X_val_fold)[:, 1]
    oof_proba[idx_val_fold] = proba_fold
    auc_fold = roc_auc_score(y_val_fold, proba_fold)

    proba_kaggle_fold = modelo_fold.predict_proba(X_kaggle)[:, 1]
    kaggle_proba_por_fold.append(proba_kaggle_fold)

    print(f"Fold {fold+1}/5 — melhor iteração: {modelo_fold.get_best_iteration():>4}  AUC-ROC = {auc_fold:.4f}")

auc_oof = roc_auc_score(y_full, oof_proba)
print(f"\nAUC-ROC out-of-fold (100 mil linhas, one-hot): {auc_oof:.4f}")
print(f"Referência (split único v2, notebook 11): 0.8263 | Kaggle público = 0.7370")


Fold 1/5 — melhor iteração:  440  AUC-ROC = 0.8326


Fold 2/5 — melhor iteração:  244  AUC-ROC = 0.8260


Fold 3/5 — melhor iteração:  312  AUC-ROC = 0.8292


Fold 4/5 — melhor iteração:  570  AUC-ROC = 0.8244


Fold 5/5 — melhor iteração:  283  AUC-ROC = 0.8177

AUC-ROC out-of-fold (100 mil linhas, one-hot): 0.8259
Referência (split único v2, notebook 11): 0.8263 | Kaggle público = 0.7370


## 3. Gerar submissão com a média dos 5 folds

In [3]:
proba_kaggle_bagging = np.mean(kaggle_proba_por_fold, axis=0)

submissao = pd.DataFrame({"Id": kaggle[ID_COL], "probabilidade_churn": proba_kaggle_bagging})
submissao.to_csv("submissions/submission_bagging_onehot_5fold.csv", index=False)
print(f"AUC-ROC out-of-fold: {auc_oof:.4f}")
print("Salvo: submissions/submission_bagging_onehot_5fold.csv")


AUC-ROC out-of-fold: 0.8259
Salvo: submissions/submission_bagging_onehot_5fold.csv
